# Segmenting remote sensing imagery with point prompts

[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeos/segment-geospatial/blob/main/docs/examples/sam2_point_prompts.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/sam2_point_prompts.ipynb)

This notebook shows how to generate object masks from point prompts with the Segment Anything Model 2 (SAM 2).

Make sure you use GPU runtime for this notebook. For Google Colab, go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

## Install dependencies

Uncomment and run the following cell to install the required dependencies.

In [1]:
%pip install segment-geospatial[samgeo2]

In [2]:
%pip install localtileserver

In [3]:
%pip install leafmap

In [4]:
%pip install -U segment-geospatial

## Import libraries

In [5]:
import leafmap
from samgeo import SamGeo2
from samgeo.common import regularize

## Create an interactive map

In [6]:
m = leafmap.Map(center=[47.653287, -117.588070], zoom=16, height="800px")
m.add_basemap("Satellite")
m

Map(center=[47.653287, -117.58807], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Download a sample image

Pan and zoom the map to select the area of interest. Use the draw tools to draw a polygon or rectangle on the map. If no geometry is drawn, the default bounding box will be used.

In [7]:
if m.user_roi is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [-117.6029, 47.65, -117.5936, 47.6563]

In [8]:
image = "satellite.tif"
leafmap.map_tiles_to_geotiff(
    output=image, bbox=bbox, zoom=18, source="Satellite", overwrite=True
)

Downloaded image 1/56
Downloaded image 2/56
Downloaded image 3/56
Downloaded image 4/56
Downloaded image 5/56
Downloaded image 6/56
Downloaded image 7/56
Downloaded image 8/56
Downloaded image 9/56
Downloaded image 10/56
Downloaded image 11/56
Downloaded image 12/56
Downloaded image 13/56
Downloaded image 14/56
Downloaded image 15/56
Downloaded image 16/56
Downloaded image 17/56
Downloaded image 18/56
Downloaded image 19/56
Downloaded image 20/56
Downloaded image 21/56
Downloaded image 22/56
Downloaded image 23/56
Downloaded image 24/56
Downloaded image 25/56
Downloaded image 26/56
Downloaded image 27/56
Downloaded image 28/56
Downloaded image 29/56
Downloaded image 30/56
Downloaded image 31/56
Downloaded image 32/56
Downloaded image 33/56
Downloaded image 34/56
Downloaded image 35/56
Downloaded image 36/56
Downloaded image 37/56
Downloaded image 38/56
Downloaded image 39/56
Downloaded image 40/56
Downloaded image 41/56
Downloaded image 42/56
Downloaded image 43/56
Downloaded image 44/

You can also use your own image. Uncomment and run the following cell to use your own image.

In [ ]:
image = '/content/mosaic_z19_2816x2816.tif'

Display the downloaded image on the map.

In [ ]:
m.layers[-1].visible = False
m.add_raster(image, layer_name="Image")
m

In [29]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [28]:
from google.colab import output
output.disable_custom_widget_manager()

## Initialize SAM class

Set `automatic=False` to enable the `SAM2ImagePredictor`.

In [ ]:
sam = SamGeo2(
    model_id="sam2-hiera-large",
    automatic=False,
)

Specify the image to segment.

In [ ]:
sam.set_image(image)

## Segment the image

Use the `predict_by_points()` method to segment the image with specified point coordinates. You can use the draw tools to add place markers on the map. If no point is added, the default sample points will be used.


In [ ]:
if m.user_rois is not None:
    point_coords_batch = m.user_rois
else:
    point_coords_batch = [
        [-117.599896, 47.655345],
        [-117.59992, 47.655167],
        [-117.599928, 47.654974],
        [-117.599518, 47.655337],
    ]

Segment the objects using the point prompts and save the output masks.

In [14]:
sam.predict_by_points(
    point_coords_batch=point_coords_batch,
    point_crs="EPSG:4326",
    output="mask.tif",
    dtype="uint8",
)

## Display the result

Add the segmented image to the map.

In [15]:
m.add_raster("mask.tif", cmap="viridis", nodata=0, opacity=0.7, layer_name="Mask")
m

Map(center=[47.65315, -117.59825000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

![image](https://github.com/user-attachments/assets/49e413b9-e159-4d72-bf23-a0318bc82d44)

## Use an existing vector dataset as points prompts

Alternatively, you can specify a file path or HTTP URL to a vector dataset containing point geometries.

In [16]:
geojson = "https://github.com/opengeos/datasets/releases/download/places/wa_building_centroids.geojson"

Display the vector data on the map.

In [17]:
m = leafmap.Map()
m.add_raster(image, layer_name="Image")
m.add_circle_markers_from_xy(
    geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)
m

Map(center=[47.65315, -117.59825000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

![image](https://github.com/user-attachments/assets/f0d3ff1e-15fa-4bd3-ac15-637e8d63527d)

## Segment image with a vector dataset

Segment the image using the specified file path to the vector dataset.

In [18]:
output_masks = "building_masks.tif"

In [19]:
sam.predict_by_points(
    point_coords_batch=geojson,
    point_crs="EPSG:4326",
    output=output_masks,
    dtype="uint8",
    multimask_output=False,
)

Display the segmented masks on the map.

In [20]:
m.add_raster(
    output_masks, cmap="jet", nodata=0, opacity=0.7, layer_name="Building masks"
)
m

Map(center=[47.65315, -117.59825000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

![image](https://github.com/user-attachments/assets/262e1a31-1648-47d2-9e71-c85ab15b1a5c)

## Clean up the result

Remove small objects from the segmented masks, fill holes, and compute geometric properties.

In [21]:
out_vector = "building_vector.geojson"
out_image = "buildings.tif"

In [22]:
array, gdf = sam.region_groups(
    output_masks, min_size=200, out_vector=out_vector, out_image=out_image
)

In [23]:
gdf.head()

,geometry,label,area,area_bbox,area_convex,area_filled,axis_major_length,axis_minor_length,eccentricity,equivalent_diameter_area,extent,orientation,perimeter,solidity,elongation
0,"POLYGON ((-13090971.338 6049795.403, -13090971...",1,1774.0,1944.0,1816.0,1774.0,58.895639,39.335134,0.744270,47.526066,0.912551,0.054659,163.112698,0.976872,1.497278
1,"POLYGON ((-13090879.99 6049795.403, -13090879....",2,1878.0,2035.0,1910.0,1878.0,59.298140,41.181848,0.719504,48.899324,0.922850,0.049432,165.941125,0.983246,1.439910
7,"POLYGON ((-13091096.12 6049794.805, -13091096....",3,1804.0,1925.0,1830.0,1804.0,60.199043,39.122954,0.760025,47.926236,0.937143,0.077096,164.284271,0.985792,1.538714
6,"POLYGON ((-13091034.027 6049794.805, -13091034...",4,1750.0,1944.0,1799.0,1750.0,57.765236,39.539690,0.729023,47.203487,0.900206,0.042879,162.769553,0.972763,1.460943
5,"POLYGON ((-13091121.195 6049792.416, -13091121...",5,1779.0,2100.0,1853.0,1779.0,54.361282,42.936259,0.613324,47.592995,0.847143,-0.253021,164.426407,0.960065,1.266093


![image](https://github.com/user-attachments/assets/af9ffa11-8ebe-4b42-8cba-3f5bcc4912f4)

## Regularize building footprints

Regularize the building footprints using the `regularize()` method.

In [24]:
output_regularized = "building_regularized.geojson"
regularize(out_vector, output_regularized)

,label,area,area_bbox,area_convex,area_filled,axis_major_length,axis_minor_length,eccentricity,equivalent_diameter_area,extent,orientation,solidity,elongation,geometry
0,1,1774.0,1944.0,1816.0,1774.0,58.895639,39.335134,0.744270,47.526066,0.912551,0.054659,0.976872,1.497278,"POLYGON ((-13090971.345 6049795.405, -13090965..."
1,2,1878.0,2035.0,1910.0,1878.0,59.298140,41.181848,0.719504,48.899324,0.922850,0.049432,0.983246,1.439910,"POLYGON ((-13090879.999 6049795.345, -13090868..."
2,8,1651.0,1748.0,1669.0,1651.0,50.799949,42.287910,0.554115,45.848866,0.944508,0.021925,0.989215,1.201288,"POLYGON ((-13091005.378 6049790.024, -13090989..."
3,6,1771.0,1927.0,1798.0,1771.0,51.205469,44.793621,0.484517,47.485864,0.919045,-0.052632,0.984983,1.143142,"POLYGON ((-13090943.191 6049790.474, -13090938..."
4,10,1392.0,1512.0,1420.0,1392.0,45.093114,40.260509,0.450393,42.099281,0.920635,0.113786,0.980282,1.120033,"POLYGON ((-13090907.465 6049787.516, -13090898..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,234,2265.0,3420.0,2318.0,2265.0,57.940265,50.427534,0.492457,53.701840,0.662281,-0.494305,0.977135,1.148981,"POLYGON ((-13090737.885 6048888.085, -13090732..."
235,236,1849.0,2193.0,1907.0,1849.0,53.288976,45.035846,0.534569,48.520304,0.843137,0.002583,0.969586,1.183257,"POLYGON ((-13090782.076 6048881.364, -13090778..."
236,238,4188.0,4756.0,4475.0,4188.0,133.077466,42.257861,0.948244,73.022786,0.880572,-1.563971,0.935866,3.149177,"POLYGON ((-13090986.857 6048874.291, -13090975..."
237,237,2140.0,2726.0,2261.0,2140.0,63.281850,44.047772,0.717987,52.198972,0.785033,1.297763,0.946484,1.436664,"POLYGON ((-13090551.025 6048876.141, -13090538..."


Display the regularized building footprints on the map.

In [25]:
m = leafmap.Map()
m.add_raster(image, layer_name="Image")
style = {
    "color": "#ffff00",
    "weight": 2,
    "fillColor": "#7c4185",
    "fillOpacity": 0,
}
m.add_raster(out_image, cmap="tab20", opacity=0.7, nodata=0, layer_name="Buildings")
m.add_vector(
    output_regularized, style=style, layer_name="Building regularized", info_mode=None
)
m

Map(center=[47.65315, -117.59825000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

![image](https://github.com/user-attachments/assets/b39ee029-2089-45b8-8ac0-ba0d750cec22)

## Interactive segmentation

In [26]:
sam.show_map()

Map(center=[47.65315, -117.59825000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…